In [14]:
import importlib
import pandas as pd
import shared.br_pv_helpers as br_pv
importlib.reload(br_pv)
# using non-checked-in data from https://zenodo.org/records/21511487
metadata_df = br_pv.get_metadata_df()
measurements_df = br_pv.get_measurements_df()
multi_location_weather_df = br_pv.get_multi_location_forecast_df()

In [17]:
import numpy as np

# ... where correlation data will be saved for each asset
irradiance_corr = {}
temperature_corr = {}
# more important: best locations. THey tell us which weather data to use for our prediction
best_locations = {}
best_loc_indices = {}

locations = br_pv.load_forecast_locations()


for ps_id in multi_location_weather_df.index.get_level_values("ps_id").unique():
    test_locations = locations[ps_id]
    test_measurements_df = filter_for_ps_id(measurements_df, ps_id).join(filter_for_ps_id(multi_location_weather_df, ps_id))
    
    irradiance_col_names = ['ghi_irradiance_wm2']
    temperature_col_names = ['ambient_temperature_celsius']
    for lid, location in enumerate(test_locations):
        irradiance_col_names.append('global_solar_irradiance_' + str(lid))
        temperature_col_names.append('temperature_' + str(lid))
    test_irradiance_df = test_measurements_df[irradiance_col_names]
    test_temperature_df = test_measurements_df[temperature_col_names]
    
    irradiance_corr[ps_id] = test_irradiance_df.corr()
    temperature_corr[ps_id] = test_temperature_df.corr()
    # avoid finding out correlation with same column is highest ;) 
    irradiance_corr[ps_id].loc['ghi_irradiance_wm2', 'ghi_irradiance_wm2'] = 0
    temperature_corr[ps_id].loc['ambient_temperature_celsius', 'ambient_temperature_celsius'] = 0
    #display(irradiance_corr[ps_id])
    #print(irradiance_corr[ps_id]['ghi_irradiance_wm2'].idxmax())
    #display(temperature_corr[ps_id])
    #print(temperature_corr[ps_id]['ambient_temperature_celsius'].idxmax())
    best_index = str(irradiance_corr[ps_id]['ghi_irradiance_wm2'].idxmax()).split('_')[-1]
    best_loc_indices[ps_id] = best_index
    best_locations[ps_id] = test_locations[str(best_index)]

print("Best locations", best_locations)

Best locations {'PS_001': {'longitude': -47.2, 'latitude': -21.6}, 'PS_002': {'longitude': -43.6, 'latitude': -22.5}, 'PS_003': {'longitude': -43.6, 'latitude': -22.5}, 'PS_004': {'longitude': -47.1, 'latitude': -23.400000000000002}, 'PS_005': {'longitude': -48.300000000000004, 'latitude': -17.1}, 'PS_006': {'longitude': -47.300000000000004, 'latitude': -22.900000000000002}, 'PS_007': {'longitude': -43.6, 'latitude': -22.5}, 'PS_008': {'longitude': -50.1, 'latitude': -21.6}, 'PS_009': {'longitude': -50.1, 'latitude': -21.6}, 'PS_010': {'longitude': -47.1, 'latitude': -23.400000000000002}, 'PS_011': {'longitude': -50.1, 'latitude': -21.6}, 'PS_012': {'longitude': -53.300000000000004, 'latitude': -17.1}, 'PS_013': {'longitude': -47.2, 'latitude': -21.6}, 'PS_015': {'longitude': -47.300000000000004, 'latitude': -17.1}, 'PS_028': {'longitude': -47.2, 'latitude': -21.6}, 'PS_033': {'longitude': -47.2, 'latitude': -21.6}, 'PS_034': {'longitude': -50.1, 'latitude': -21.6}, 'PS_035': {'longitu

In [20]:
# export best locations
best_location_dict = {
    "lon": {},
    "lat": {},
    "loc_id": {}
}
for ps_id, location in best_locations.items():
    best_location_dict['loc_id'][ps_id] = best_loc_indices[ps_id]
    best_location_dict['lat'][ps_id] = location['latitude']
    best_location_dict['lon'][ps_id] = location['longitude']

best_location_df = pd.DataFrame(best_location_dict)
best_location_df.to_parquet(path + "best_locations.parquet")
best_location_df

,lon,lat,loc_id
PS_001,-47.2,-21.6,10
PS_002,-43.6,-22.5,5
PS_003,-43.6,-22.5,5
PS_004,-47.1,-23.4,6
PS_005,-48.3,-17.1,6
PS_006,-47.3,-22.9,1
PS_007,-43.6,-22.5,5
PS_008,-50.1,-21.6,9
PS_009,-50.1,-21.6,9
PS_010,-47.1,-23.4,6
